In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np

In [ ]:
k = 0
p = 0.05
exp = "OS"
obs="O"
results = pd.read_csv(f'results/{k}/{p}/exp_{exp}.csv', index_col=0)
sns.set_style('whitegrid')
results['method_'] = results['method'] + ' ' + results['k_inv'].astype(str)
results['method_'] = results['method_'].replace(' nan', '', regex=True)
ATE = results["RCT"].mean()
results = results[results['method_'].isin(["DERM", "ERM"])]
# rename methods_ values
results['method_'] = results['method_'].replace('CURL 0.1', 'Ours').replace('UCRL 0.1', 'v-REx')
#results = results[results['method_'].isin(["CURL 0.1", "CURL+ 0.1", "UCRL 0.1", "ERM"])]

#results = results[results['method']=="UCRL"]
# l = np.exp(results.loc[results["method"]=="CURL", "train_ratio"]-1)
# results.loc[results["method"]=="CURL", "AIPW_"] = results.loc[results["method"]=="CURL", "AIPW_"]*(1-l) + results.loc[results["method"]=="CURL", "AIPW_tr"]*(l)
results["TERB"] = (results[f"OS{obs}C_"] - ATE)/ATE*100


plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.lineplot(x='train_ratio', y='TERB', hue='method_', data=results)
sns.lineplot(x=np.array(results.loc[results["method"]=="ERM", "train_ratio"]), 
             y=np.array((results.loc[results["method"]=="ERM", f"OS{obs}C_tr"]-ATE)/ATE*100),
             label='AIPW (train)')
plt.title(f'k={k}, p={p}, exp={exp}')
plt.axhline(y=0, color='black', linestyle='--')#, label='ATE')
plt.xlabel('Train Ratio')
plt.ylabel('TERB')
plt.xscale('log')
plt.xticks(results['train_ratio'].unique(), results['train_ratio'].unique())
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.legend(title='Method')

plt.subplot(1, 2, 2)
sns.lineplot(x='train_ratio', y='val_acc', hue='method_', data=results)
plt.title(f'k={k}, p={p}, exp={exp}')
plt.xlabel('Train Ratio')
plt.ylabel('Validation Accuracy')
plt.xscale('log')
plt.xticks(results['train_ratio'].unique(), results['train_ratio'].unique())
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.legend(title='Method')

plt.savefig(f'results/plots/k_{k}_p_{p}_exp_{exp}_{obs}.png')
plt.show()

In [ ]:
# import generalization results 
pW=0.5
pU=0.02
e=1
exp = "RCT"
results = pd.read_csv(f'results/{e}/{pW}/{pU}/{exp}/generalization.csv', index_col=0)
results['bias'] = results['ATE'] - results['PPAIPW_OC']
seeds = results[((results['method']=='DERM') & (results['target']=='RT') & (results['acc']<0.95))]['seed']
results = results[~(results['seed'].isin(seeds) & (results['method']=='DERM'))]
results_grouped = results.drop(['seed'], axis=1).groupby(['method', 'target']).agg(['mean', 'std'])
results_grouped
results_grouped['PPAIPW_OC']


In [ ]:
# print scatterplot Bias vs Acc coloring by method and shaping by target
plt.figure(figsize=(8, 6))
sns.scatterplot(x='bias', y='acc', hue='method', data=results, style='target')
plt.xlabel('Bias')
plt.ylabel('Accuracy');

In [ ]:
# import generalization results 
k=2
p=0.9
results = pd.read_csv(f'results/{k}/{p}/generalization.csv', index_col=0)
results = results.drop(columns=['k_inv', 'exp', 'seed'])
# group by mean and variance
results = results.groupby(['method', 'test']).agg(['mean', 'std']).reset_index()
results = results[["method","test","tr_acc","OSOC_","RCT"]]
results = results.rename(columns={"tr_acc":"Accuracy", "OSOC_":"PP-ATE", "RCT":"ATE"})
#results["Bias"] = results["PP-ATE"] - results["ATE"]
results

In [ ]:
k = 2
p = 0.3
exp = "OS"
results = pd.read_csv(f'results/{k}/{p}/exp_{exp}.csv', index_col=0)
sns.set_style('whitegrid')
results['method_'] = results['method'] + ' ' + results['k_inv'].astype(str)
results['method_'] = results['method_'].replace(' nan', '', regex=True)
ATE = results["ATE"].mean()
results = results[results['method_'].isin(["CURL 0.1", "UCRL 0.1", "ERM"])]
#results = results[results['method_'].isin(["CURL 0.1", "CURL+ 0.1", "UCRL 0.1", "ERM"])]

#results = results[results['method']=="UCRL"]
# l = np.exp(results.loc[results["method"]=="CURL", "train_ratio"]-1)
# results.loc[results["method"]=="CURL", "AIPW_"] = results.loc[results["method"]=="CURL", "AIPW_"]*(1-l) + results.loc[results["method"]=="CURL", "AIPW_tr"]*(l)
results["TERB"] = (results["AIPW_"] - ATE)/ATE*100


plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.lineplot(x='train_ratio', y='TERB', hue='method_', data=results)
sns.lineplot(x=np.array(results.loc[results["method"]=="ERM", "train_ratio"]), 
             y=np.array((results.loc[results["method"]=="ERM", "AIPW_tr"]-ATE)/ATE*100),
             label='AIPW (train)')
plt.title(f'k={k}, p={p}, exp={exp}')
plt.axhline(y=0, color='black', linestyle='--')#, label='ATE')
plt.xlabel('Train Ratio')
plt.ylabel('TERB')
plt.xscale('log')
plt.xticks(results['train_ratio'].unique(), results['train_ratio'].unique())
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.legend(title='Method')

plt.subplot(1, 2, 2)
sns.lineplot(x='train_ratio', y='val_acc', hue='method_', data=results)
plt.title(f'k={k}, p={p}, exp={exp}')
plt.xlabel('Train Ratio')
plt.ylabel('Validation Accuracy')
plt.xscale('log')
plt.xticks(results['train_ratio'].unique(), results['train_ratio'].unique())
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.legend(title='Method')

plt.savefig(f'results/plots/k_{k}_p_{p}_exp_{exp}.png')
plt.show()

In [ ]:
# plot AIPW_tr vs train_ratio
plt.figure(figsize=(16, 6))
sns.lineplot(x='train_ratio', y='AIPW_', hue='method_', data=results)
plt.title(f'k={k}, p={p}, exp={exp}')
plt.xlabel('Train Ratio')
plt.ylabel('AIPW')
plt.xscale('log')
plt.xticks(results['train_ratio'].unique(), results['train_ratio'].unique())
plt.legend(title='Method')
# add hlife for AIPW mean
plt.axhline(y=ATE, color='black', linestyle='--', label='ATE')
plt.show()
